# Business Rule: FX, Balances, Fraud and AML

Enriches the validated silver transactions with business rules and writes the results to the silver and gold layers.

- Convert amounts to INR and USD using daily FX rates (gaps filled with the last known rate)
- Signed cash flow, running balance and day-closing balance per account
- Transaction category from the MCC reference, with channel-based fallbacks
- Fraud and AML flags (velocity, structuring, high value / high-risk MCC / odd hours / round amounts, dormant accounts) and a weighted risk score

**Reads:** `silver.silver_transactions`, `silver.silver_fx_rates`, `bronze.bronze_mcc_codes`  
**Writes:** `silver.silver_transaction_enriched` and the gold tables `gold_account_running_position`, `gold_account_daily_position`, `gold_transaction_velocity_flags`, `gold_transaction_structuring_flags`, `gold_transaction_risk_flags`, `gold_dormant_activity_flags`, `gold_transaction_risk_score` and `gold_fact_transaction`  
**Parameters:** `AS_OF`, `BATCH_ID`

## Imports

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Shared functions

In [ ]:
%run NB_FUNCTIONS

## Parameters

The next cell is tagged `parameters`, so a pipeline can override it at run time.

In [ ]:
AS_OF = ""
BATCH_ID = ""

## Load source tables

In [ ]:
# importing silver_fx_rates dataframe
fx = spark.read.table("silver.silver_fx_rates")

# importing silver_transaction dataframe for reference
txn_valid = spark.read.table("silver.silver_transactions")

# importing mcc_code dataframe
mcc_code_ref = (
    spark.read.table("bronze.bronze_mcc_codes")
    .withColumn("mcc_high_risk", F.col("is_high_risk") == "Y")
    .select("mcc", "category", "mcc_high_risk")
    .drop("is_high_risk")
)

### Standardizing Calendar Rate Dates

In [ ]:
calendar = fx.agg(
    F.min("rate_date").alias("low"), F.lit(AS_OF).cast("date").alias("high")
).select(F.explode(F.sequence("low", "high")).alias("rate_date"))

# window
w = (
    Window.partitionBy("from_ccy")
    .orderBy("rate_date")
    .rowsBetween(Window.unboundedPreceding, 0)
)

fx_daily = (
    calendar.crossJoin(fx.select("from_ccy").distinct())
    .join(fx.select("rate_date", "from_ccy", "rate"), ["rate_date", "from_ccy"], "left")
    .withColumn("is_filled", F.col("rate").isNull())
    .withColumn("rate", F.last("rate", ignorenulls=True).over(w))
    .unionByName(
        calendar.select(
            "rate_date",
            F.lit("INR").alias("from_ccy"),
            F.lit(1).cast("decimal(18,6)").alias("rate"),
            F.lit(False).alias("is_filled"),
        )
    )
)

fx_daily.groupBy("from_ccy").agg(
    F.count("*").alias("count"),
    F.sum(F.col("is_filled").cast("int")).alias("count_of_filled"),
).show()

## Converting amount to USD and INR

In [ ]:
# filtering USD rate's for every rate date
usd = fx_daily.filter(F.col("from_ccy") == "USD").select(
    "rate_date", F.col("rate").alias("usd_inr")
)

# joining the rate of all particular date with customers transactions
txn_fx = (
    txn_valid.join(
        F.broadcast(
            fx_daily.withColumnRenamed("from_ccy", "currency").withColumnRenamed(
                "rate_date", "txn_date_ist"
            )
        ),
        ["currency", "txn_date_ist"],
        "left",
    )
    .join(
        F.broadcast(usd.withColumnRenamed("rate_date", "txn_date_ist")),
        "txn_date_ist",
        "left",
    )
    .withColumn(
        "amount_inr", F.round(F.col("amount") * F.col("rate"), 2).cast("decimal(18,2)")
    )
    .withColumn(
        "amount_usd",
        F.round(F.col("amount_inr") / F.col("usd_inr"), 2).cast("decimal(10,2)"),
    )
    .drop("is_filled")
)

In [ ]:
print(
    f"Number of transaction with Zero amount in INR: {txn_fx.filter('amount_inr IS NULL').count()}"
)

## Net Cash Flow
- Debit means flow is OUT
- Credit means flow is IN
- Signed Amount (-ve & +ve)
    - Failed transaction = 0
    - Success Transaction + Debit = -ve amount
    - Success Transaction + Credit = +ve amount

In [ ]:
# Net Cash Flow and Direction
txn_fx = txn_fx.withColumn(
    "direction", F.when(F.col("txn_type") == "DEBIT", "OUT").otherwise("IN")
).withColumn(
    "signed_amount_inr",
    F.when(F.col("status") != "SUCCESS", F.lit(0).cast("decimal(18,2)"))
    .when(F.col("txn_type") == "DEBIT", -F.col("amount_inr"))
    .otherwise("amount_inr"),
)

## Categorizing Transaction

In [ ]:
txn_fx = txn_fx.join(F.broadcast(mcc_code_ref), "mcc", "left").withColumn(
    "category",
    F.coalesce(
        "category",
        F.when(F.col("channel") == "ATM", "Cash Withdrawal")
        .when(F.col("channel").isin("UPI", "NEFT", "IMPS"), "Transfers")
        .otherwise("Uncategorised"),
    ),
)

txn_fx.groupBy("category").count().orderBy(F.desc("count")).show(20)

## Running Balance

In [ ]:
w_run = (
    Window.partitionBy("account_id")
    .orderBy("txn_ts", "txn_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
txn_fx = txn_fx.withColumn("running_net_inr", F.sum("signed_amount_inr").over(w_run))

In [ ]:
display(txn_fx)

In [ ]:
running_position = txn_fx.select(
    "txn_id",
    "account_id",
    "customer_id",
    "account_type",
    "txn_date_ist",
    "currency",
    "amount_inr",
    "amount_usd",
    "signed_amount_inr",
    "running_net_inr",
    "txn_type",
    "channel",
    "status",
    "direction",
    "category",
    "mcc_high_risk",
    "_batch_id",
)

# Routing to Gold Layer
delta_merge(
    df=running_position,
    schema="gold",
    table_name="gold_account_running_position",
    merge_cond="target.txn_id = source.txn_id",
)

## Day Closing Balance Per Account

In [ ]:
w_last = Window.partitionBy("account_id", "txn_date_ist").orderBy(
    F.desc("txn_ts"), F.desc("txn_id")
)
daily_position = (
    txn_fx.withColumn("_rn", F.row_number().over(w_last))
    .filter("_rn = 1")
    .select(
        "account_id", "txn_date_ist", F.col("running_net_inr").alias("closing_net_inr")
    )
)

In [ ]:
# Routing to Gold Layer
delta_merge(
    df=daily_position,
    schema="gold",
    table_name="gold_account_daily_position",
    merge_cond="target.account_id = source.account_id AND target.txn_date_ist = source.txn_date_ist",
)

## Fraud Check/Test

## Fraud rule 1: Velocity (many card transactions in 10 minutes)

In [ ]:
w_10m = (
    Window.partitionBy("account_id")
    .orderBy(F.col("txn_ts").cast("long"))
    .rangeBetween(-600, 0)
)

txn_fx = txn_fx.withColumn(
    "txn_count_10m",
    F.sum(F.when(F.col("card_last4").isNotNull(), 1).otherwise(0)).over(w_10m),
).withColumn(
    "flag_velocity",
    F.coalesce(
        (F.col("card_last4").isNotNull()) & (F.col("txn_count_10m") >= 5), F.lit(False)
    ),
)

In [ ]:
print(
    f"Fraud Alert! Number of Account Flagged for Fraud: {txn_fx.filter('flag_velocity').select('account_id').distinct().count()}"
)

In [ ]:
# routing to gold table
velocity_flags = txn_fx.filter(F.col("flag_velocity") == True).select(
    "txn_id",
    "account_id",
    "customer_id",
    "txn_ts",
    "amount_inr",
    "amount_usd",
    "currency",
    "txn_type",
    "channel",
    "card_last4",
    "txn_count_10m",
    "flag_velocity",
    "_batch_id",
)

delta_merge(
    df=velocity_flags,
    schema="gold",
    table_name="gold_transaction_velocity_flags",
    merge_cond="target.txn_id = source.txn_id",
)

## Fraud rule 2: Money laundering (large amounts split into several transfers under 50,000 INR)

In [ ]:
near_limit = (
    (F.col("txn_type") == "DEBIT")
    & F.col("channel").isin("IMPS", "NEFT", "UPI")
    & (F.col("amount_inr") >= 45000)
    & (F.col("amount_inr") < 50000)
)

w_24h = (
    Window.partitionBy("account_id")
    .orderBy(F.col("txn_ts").cast("long"))
    .rangeBetween(-86400, 0)
)


txn_fx = (
    txn_fx.withColumn("_near", near_limit.cast("int"))
    .withColumn("near_limit_24h", F.sum("_near").over(w_24h))
    .withColumn("flag_structuring", near_limit & (F.col("near_limit_24h") >= 3))
    .drop("_near")
)

print(
    f"Number of accounts flagged for suspicious Transactions: {txn_fx.filter('flag_structuring').select('account_id').distinct().count()}"
)

In [ ]:
structuring_flags = txn_fx.filter(F.col("flag_structuring") == True).select(
    "txn_id",
    "account_id",
    "customer_id",
    "txn_ts",
    "amount_inr",
    "txn_type",
    "channel",
    "near_limit_24h",
    "flag_structuring",
    "_batch_id",
)

delta_merge(
    df=structuring_flags,
    schema="gold",
    table_name="gold_transaction_structuring_flags",
    merge_cond="target.txn_id = source.txn_id",
)

## Fraud rule 3: Odd time, high amount (high value, high-risk merchants, odd hours)
- Transactions of ₹2,00,000 or more.
- International spending at high-risk merchants( _ref. mcc_code df_ ) like gambling, crypto, or wire transfers, unusually large round figures.
- Any activity occurring between midnight and 5 am IST.

In [ ]:
# current IST timezone timestamp
hour_ist = F.hour(F.from_utc_timestamp("txn_ts", "Asia/Kolkata"))
txn_fx = (
    txn_fx.withColumn("flag_high_value", F.col("amount_inr") >= 200000)
    .withColumn(
        "flag_high_risk_mcc", F.col("mcc_high_risk") & (F.col("txn_type") == "DEBIT")
    )
    .withColumn(
        "flag_round_amount",
        (F.col("amount_inr") >= 10000) & (F.col("amount_inr") % 10000 == 0),
    )
    .withColumn("flag_odd_hour", hour_ist < 5)
)

print(
    txn_fx.filter(
        "flag_high_value OR flag_high_risk_mcc OR flag_round_amount OR flag_odd_hour"
    )
    .select("account_id")
    .distinct()
    .count()
)

In [ ]:
risk_flags = txn_fx.filter(
    F.col("flag_high_value")
    | F.col("flag_high_risk_mcc")
    | F.col("flag_round_amount")
    | F.col("flag_odd_hour")
).select(
    "txn_id",
    "account_id",
    "customer_id",
    "txn_ts",
    "amount_inr",
    "currency",
    "txn_type",
    "channel",
    "category",
    "mcc",
    "mcc_high_risk",
    "flag_high_value",
    "flag_high_risk_mcc",
    "flag_round_amount",
    "flag_odd_hour",
    "_batch_id",
)

delta_merge(
    df=risk_flags,
    schema="gold",
    table_name="gold_transaction_risk_flags",
    merge_cond="target.txn_id = source.txn_id",
)

## Fraud rule 5: Dormant account (sudden transaction by a dormant account)

In [ ]:
txn_fx = txn_fx.withColumn(
    "flag_dormant_activity",
    (F.col("account_status") == "DORMANT")
    & (F.col("txn_type") == "DEBIT")
    & (F.col("status") == "SUCCESS"),
)

print(txn_fx.filter("flag_dormant_activity").count())

In [ ]:
dormant_flags = txn_fx.filter(F.col("flag_dormant_activity") == True).select(
    "txn_id",
    "account_id",
    "customer_id",
    "txn_ts",
    "amount_inr",
    "currency",
    "txn_type",
    "status",
    "account_status",
    "channel",
    "flag_dormant_activity",
    "_batch_id",
)

delta_merge(
    df=dormant_flags,
    schema="gold",
    table_name="gold_dormant_activity_flags",
    merge_cond="target.txn_id = source.txn_id",
)

## Weigh Risk Band
Score Points are requested from requester!

In [ ]:
WEIGHTS = {
    "flag_velocity": 40,
    "flag_structuring": 50,
    "flag_high_value": 20,
    "flag_high_risk_mcc": 25,
    "flag_dormant_activity": 35,
    "flag_odd_hour": 5,
    "flag_round_amount": 5,
}
score = sum(F.when(F.col(c), w).otherwise(0) for c, w in WEIGHTS.items())

txn_fx = txn_fx.withColumn("risk_score", score).withColumn(
    "risk_band",
    F.when(F.col("risk_score") >= 50, "HIGH")
    .when(F.col("risk_score") >= 25, "MEDIUM")
    .otherwise("LOW"),
)

txn_fx.groupBy("risk_band").count().show()

In [ ]:
total_risk_score_df = txn_fx.select(
    "txn_id",
    "account_id",
    "customer_id",
    "txn_ts",
    "amount_inr",
    "currency",
    "risk_score",
    "risk_band",
    "flag_velocity",
    "flag_structuring",
    "flag_high_value",
    "flag_high_risk_mcc",
    "flag_round_amount",
    "flag_odd_hour",
    "flag_dormant_activity",
    "_batch_id",
)


delta_merge(
    df=total_risk_score_df,
    schema="gold",
    table_name="gold_transaction_risk_score",
    merge_cond="target.txn_id = source.txn_id",
)

In [ ]:
display(txn_fx.filter(F.col("risk_band") == "HIGH"))

In [ ]:
# saving it to silver layer!
delta_merge(
    df=txn_fx,
    schema="silver",
    table_name="silver_transaction_enriched",
    merge_cond="target.txn_id = source.txn_id",
)

## Enriched Transaction Table To Gold Layer

In [ ]:
fact_cols = [
    "txn_id",
    "account_id",
    "customer_id",
    "txn_ts",
    "txn_date_ist",
    "txn_type",
    "channel",
    "status",
    "currency",
    "amount",
    "amount_inr",
    "amount_usd",
    "signed_amount_inr",
    "category",
    "merchant_name",
    "merchant_location_city",
    "is_international",
    "risk_score",
    "risk_band",
]

fact_transaction = txn_fx.select(*fact_cols)
# routing it to gold layer
delta_merge(
    df=fact_transaction,
    schema="gold",
    table_name="gold_fact_transaction",
    merge_cond="target.txn_id = source.txn_id",
)